# Spectral time evolution in a one-dimensional box

**Prerequisite:** [Spectral time evolution in a one-dimensional box](../../../docs/lecture-notes/qm/piab1d/piab1d__time_evolution.md)

This maintained laboratory propagates a finite analytical superposition of stationary particle-in-a-box modes. The selected model is nondimensional: $L=1$, $m=1$, and $\hbar=1$.

## Learning objectives

1. Represent an initial state by normalized stationary-basis coefficients.
2. Propagate each coefficient with its exact energy phase.
3. Verify norm and energy-expectation conservation.
4. Relate interference dynamics to the energy-difference beat period.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from projectkoios.physkit.qm.piab1d.base import Piab1D
from projectkoios.physkit.qm.piab1d.tdse_analytical.solver import (
    Piab1dTdseAnalyticalSolver,
)
from projectkoios.physkit.qm.piab1d.tise_analytical.solution import (
    Piab1DAnalyticalSolution,
)
from projectkoios.physkit.units import (
    ComplexVectorQuantity,
    Unitless,
    UnitSystem,
    VectorQuantity,
)

In [ ]:
model = Piab1D(
    length=1.0,
    mass=1.0,
    unit_system=UnitSystem.NONDIMENSIONAL,
)
mode_count = 4
tise_solution = Piab1DAnalyticalSolution(model=model).evaluate(count=mode_count)

initial_coefficients = np.zeros(mode_count, dtype=np.complex128)
initial_coefficients[0] = 1.0 / np.sqrt(2.0)
initial_coefficients[1] = 1.0j / np.sqrt(2.0)

energy_difference = (
    tise_solution.energies.magnitude[1] - tise_solution.energies.magnitude[0]
)
beat_period = 2.0 * np.pi * model.hbar_quantity.magnitude / energy_difference
time_values = np.linspace(0.0, beat_period, 81, dtype=np.float64)

evolution = Piab1dTdseAnalyticalSolver().solve(
    tise_results=tise_solution,
    initial_coefficients=ComplexVectorQuantity(
        magnitude=initial_coefficients, unit=Unitless()
    ),
    times=VectorQuantity(magnitude=time_values, unit=Unitless()),
)

In [ ]:
expected_energy = float(
    0.5 * (tise_solution.energies.magnitude[0] + tise_solution.energies.magnitude[1])
)

assert np.allclose(
    evolution.norms.magnitude,
    1.0,
    rtol=0.0,
    atol=2e-15,
)
assert np.isclose(
    evolution.energy_expectation.magnitude,
    expected_energy,
    rtol=2e-15,
    atol=0.0,
)
beat_period, evolution.energy_expectation.magnitude

In [ ]:
coordinate_values = np.linspace(0.0, 1.0, 401, dtype=np.float64)
coordinates = VectorQuantity(magnitude=coordinate_values, unit=Unitless())
sampled_evolution = evolution.state_evolution(coordinates=coordinates)
state_amplitudes = np.column_stack(
    tuple(state.amplitudes.magnitude for state in sampled_evolution)
)
probability_density = np.abs(state_amplitudes) ** 2
represented_norms = np.trapezoid(
    probability_density,
    x=coordinate_values,
    axis=0,
)

assert np.allclose(
    represented_norms,
    1.0,
    rtol=0.0,
    atol=2e-15,
)
assert np.allclose(
    probability_density[:, -1],
    probability_density[:, 0],
    rtol=0.0,
    atol=2e-15,
)
float(np.max(np.abs(represented_norms - 1.0)))

In [ ]:
figure, axes = plt.subplots(1, 2, figsize=(12, 4.5))

image = axes[0].pcolormesh(
    coordinate_values,
    time_values / beat_period,
    probability_density.T,
    shading="auto",
)
axes[0].set(
    xlabel="$x/L$",
    ylabel="time / beat period",
    title="Probability-density evolution",
)
figure.colorbar(image, ax=axes[0], label=r"$|\psi(x,t)|^2$")

selected_time_indices = (0, 20, 40, 60, 80)
for index in selected_time_indices:
    axes[1].plot(
        coordinate_values,
        probability_density[:, index],
        label=rf"$t/T_{{beat}}={time_values[index] / beat_period:.2f}$",
    )
axes[1].set(
    xlabel="$x/L$",
    ylabel=r"$|\psi(x,t)|^2$",
    title="Selected probability-density profiles",
)
axes[1].grid(True)
axes[1].legend()

figure.tight_layout()
plt.show()

## Interpretation

The analytical propagator changes only the stationary-basis phases, so the spectral norm and energy expectation remain constant. The relative phase between the first two modes evolves at $(E_2-E_1)/\hbar$. After one beat period, the probability density returns to its initial profile even though the full wavefunction may differ by a global phase.

The four-mode analytical basis is sufficient here because only the first two coefficients are nonzero. A general initial spatial function requires a declared projection procedure and a controlled basis-truncation study.

## Exercises

1. Use a real equal superposition and compare the initial density with this laboratory's phase-shifted state.
2. Change the relative amplitudes while preserving coefficient normalization.
3. Add a third nonzero mode and determine whether one two-mode beat period still restores the density.
4. Increase the represented mode count while leaving the nonzero coefficient inventory unchanged.
